# Advanced RAG: Team + Mentor Recommendation (Grounded + Evidence)

This notebook implements a dataset-aligned RAG pipeline to:
- infer 2 complementary student roles + mentor expertise
- retrieve role-specific evidence pools using query decomposition + rephrasing
- recommend **2 distinct students** and **1 mentor**
- provide **2–4 evidence bullets per selection**
- fall back to: **Not found in the provided data.** when required evidence is missing


In [1]:
!pip -q install pinecone langchain langchain-openai langchain-pinecone pydantic

import os, json, re
from pathlib import Path
from typing import List, Dict, Any, Tuple
from collections import defaultdict, Counter

from pinecone import Pinecone

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.


## Environment keys & imports

You need:
- `OPENAI_API_KEY`
- `PINECONE_API_KEY`

We will connect to an **existing Pinecone index** (no re-indexing in this notebook).


In [2]:
# 🔐 Step 2: Configure Environment Variables and API Keys

from google.colab import userdata
import os

# Enable tracing for LangSmith to monitor and debug LangChain runs
os.environ["LANGSMITH_TRACING"] = "true"

# Set LangSmith API endpoint and key for experiment tracking
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')

# Assign project name for grouping all traces under one dashboard
os.environ["LANGSMITH_PROJECT"] = "RAG PROJECT"

# Retrieve OpenAI and Pinecone API keys securely from Colab’s secret storage
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["PINECONE_API_KEY"] = userdata.get('PINECONE_API_KEY')

# Define the embedding model and Pinecone configuration details
OPENAI_MODEL_EMB = "text-embedding-3-small"   # Embedding model (1536-dim output)
INDEX_NAME = "day3-rag-index"                 # Pinecone index name
PINECONE_CLOUD = "aws"                        # Cloud provider for Pinecone (AWS or GCP)
PINECONE_REGION = "us-east-1"
PINECONE_ENV = "us-east-1-aws"


LLM_MODEL = "gpt-4o-mini"


In [3]:
import os
from typing import List, Dict, Any
from pinecone import Pinecone

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# ---- PATHS ----
DATA_ROOT = Path("/content/drive/MyDrive/APU TRAINING/TRAINING DATA")  # adjust if needed
STUDENTS_DIR = DATA_ROOT / "students"

assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY"
assert os.getenv("PINECONE_API_KEY"), "Missing PINECONE_API_KEY"


#Build Student Profile Map (Code)

This uses your dataset structure: students/<StudentFolder>/metadata.json

In [4]:
def build_student_profile_map(students_dir: Path) -> Dict[str, Dict[str, Any]]:
    profile = {}
    for student_folder in sorted(students_dir.glob("*")):
        if not student_folder.is_dir():
            continue
        meta_path = student_folder / "metadata.json"
        if not meta_path.exists():
            continue
        try:
            meta = json.loads(meta_path.read_text(encoding="utf-8"))
            student_key = student_folder.name  # stable folder key
            profile[student_key] = meta
        except Exception as e:
            print(f"Failed reading {meta_path}: {e}")
    return profile

student_profile_map = build_student_profile_map(STUDENTS_DIR)
print("Loaded student profiles:", len(student_profile_map))
print("Example keys:", list(student_profile_map.keys())[:5])


Loaded student profiles: 10
Example keys: ['Ahmed', 'Aisha', 'David', 'Kenji', 'Lisa']


# Connect to Existing Pinecone Index

In [5]:

pc = Pinecone()
index = pc.Index(INDEX_NAME)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")  # must match what you used during indexing
vectorstore = PineconeVectorStore(index=index, embedding=embeddings)

# small-k retrieval per query (we'll union results across many queries)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Connected to index:", INDEX_NAME)


Connected to index: day3-rag-index


# Helper Functions: Metadata + Chunk Types

In [6]:
def get_student_key(doc: Document) -> str | None:
    """
    Returns folder-key like 'David', 'Kenji', etc.
    Adjust if your metadata uses a different field name.
    """
    md = doc.metadata or {}
    return md.get("student_key") or md.get("student")  # try both

def is_student_chunk(doc: Document) -> bool:
    md = doc.metadata or {}
    cat = (md.get("category") or "").lower()
    # allow any chunk that is clearly from students folder
    src = (md.get("source") or "").lower()
    return ("student" in cat) or ("students/" in src)

def is_mentor_chunk(doc: Document) -> bool:
    md = doc.metadata or {}
    src = (md.get("source") or "").lower()
    cat = (md.get("category") or "").lower()
    # strict mentor isolation: Mentors.pdf OR category mentor
    return ("mentors.pdf" in src) or ("mentor" == cat) or ("mentor" in cat)

def safe_text(doc: Document, limit: int = 1200) -> str:
    t = (doc.page_content or "").strip()
    return t[:limit]

def dedupe_docs(docs: List[Document]) -> List[Document]:
    seen = set()
    out = []
    for d in docs:
        key = (d.metadata.get("id") or d.metadata.get("chunk_id") or None, safe_text(d, 200))
        if key in seen:
            continue
        seen.add(key)
        out.append(d)
    return out

def clean_mentor_text(text: str) -> str:
    """
    Fix space-separated PDF extraction artifacts.
    e.g. 'Dr.\\n \\nNancy\\n \\nChen' → 'Dr. Nancy Chen'
    """
    # collapse all whitespace variants into single space
    return " ".join(text.split())


## [A] Role Inference
We infer:
- `role1` (2–5 words)
- `role2` (2–5 words)
- `mentor` expertise (2–5 words)

Output must be exactly 3 lines:
role1: ...
role2: ...
mentor: ...


In [7]:
role_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

ROLE_PROMPT = """You are a technical role classifier for student project teams.
Given a project query, output EXACTLY 3 lines with no extra text:
role1: <2-5 technical keywords for student role 1>
role2: <2-5 technical keywords for student role 2>
mentor: <2-5 technical keywords for mentor expertise>

Rules:
- role1 and role2 must be DIFFERENT and complementary
- Use domain terms from the project (e.g., computer vision, edge AI, IoT, NLP, embedded systems)
- mentor should reflect the project domain expertise
- Output ONLY the 3 lines, nothing else, no preamble

User query: {q}
"""

def infer_roles(user_query: str) -> Dict[str, str]:
    msg = ROLE_PROMPT.format(q=user_query)
    resp = role_llm.invoke(msg).content.strip()
    lines = [ln.strip() for ln in resp.splitlines() if ln.strip()]

    # Safe defaults so subqueries are NEVER empty
    out = {
        "role1": "edge deployment embedded systems",
        "role2": "computer vision monitoring pipeline",
        "mentor": "applied AI computer vision systems"
    }

    for ln in lines:
        if ":" not in ln:
            continue
        key, val = ln.split(":", 1)
        key = key.strip().lower()
        val = val.strip()
        if val and key in out:
            out[key] = val

    return out

# quick test
print(infer_roles("Road Monitoring with Raspberry Pi. Find me 2 students and a guide."))

{'role1': 'Raspberry Pi, sensors, data collection', 'role2': 'image processing, computer vision, machine learning', 'mentor': 'IoT, embedded systems, real-time data analysis'}


# [B] Decomposition (Code)

We build 3 base sub-queries by combining role labels + query keywords.

In [8]:
def tokenize_keywords(text: str) -> List[str]:
    # simple keyword extraction: keep meaningful tokens
    toks = re.findall(r"[a-zA-Z0-9]+", text.lower())
    stop = {"find","me","a","an","the","and","with","for","to","of","i","have","project","students","student","guide","mentor"}
    toks = [t for t in toks if t not in stop and len(t) > 2]
    # de-dup preserve order
    seen = set()
    out = []
    for t in toks:
        if t not in seen:
            seen.add(t)
            out.append(t)
    return out

def build_subqueries(user_query: str, roles: Dict[str, str]) -> Dict[str, str]:
    kw = tokenize_keywords(user_query)
    kw_str = " ".join(kw[:10])  # cap
    q1 = f"{roles['role1']} {kw_str}".strip()
    q2 = f"{roles['role2']} {kw_str}".strip()
    q3 = f"{roles['mentor']} {kw_str}".strip()
    return {"Q1": q1, "Q2": q2, "Q3": q3, "project_keywords": kw}

# test
uq = "Road Monitoring with Raspberry Pi. Find me 2 students and a guide."
roles = infer_roles(uq)
subs = build_subqueries(uq, roles)
roles, subs


({'role1': 'Raspberry Pi, sensors, data collection',
  'role2': 'image processing, computer vision, software development',
  'mentor': 'IoT, embedded systems, project management'},
 {'Q1': 'Raspberry Pi, sensors, data collection road monitoring raspberry',
  'Q2': 'image processing, computer vision, software development road monitoring raspberry',
  'Q3': 'IoT, embedded systems, project management road monitoring raspberry',
  'project_keywords': ['road', 'monitoring', 'raspberry']})

# [C] Query Rephrasing
For each sub-query, generate 2–3 rewritten retrieval queries.
Rules:
- One query per line
- No numbering/bullets
- Short, keyword-focused


In [9]:
rewrite_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

REWRITE_PROMPT = """Generate 3 alternative search queries for retrieval.
Rules:
- Output exactly 3 lines
- No numbering, no bullets
- Each line is a complete query (short, keyword-focused)

Base query: {q}
"""

def rephrase_queries(base_query: str) -> List[str]:
    resp = rewrite_llm.invoke(REWRITE_PROMPT.format(q=base_query)).content.strip()
    lines = [ln.strip(" -•\t").strip() for ln in resp.splitlines() if ln.strip()]
    # include original first, then rewrites, dedupe preserve order
    out = []
    for x in [base_query] + lines[:3]:
        if x and x not in out:
            out.append(x)
    return out[:4]  # original + up to 3

# test
rephrase_queries(subs["Q1"])


MENTOR_REWRITE_PROMPT = """Generate 4 mentor-focused search queries to retrieve from a Mentor list document.
Rules:
- Output exactly 4 lines
- No numbering, no bullets
- Each query MUST include at least one of these words: mentor, guide, supervision, expertise
- Keep queries short and keyword-focused

Project context: {q}
"""

def rephrase_mentor_queries(project_context: str, mentor_label: str) -> List[str]:
    base = f"{mentor_label} {project_context}".strip()
    resp = rewrite_llm.invoke(MENTOR_REWRITE_PROMPT.format(q=base)).content.strip()
    lines = [ln.strip(" -•\t").strip() for ln in resp.splitlines() if ln.strip()]
    # ensure "Mentors.pdf" anchored intent via mandatory tokens
    out = []
    for x in lines[:4]:
        if x and x not in out:
            out.append(x)
    # Add a safe fallback query
    out.append(f"mentor expertise {mentor_label} {project_context}")
    return out[:5]


# [D] Retrieval per Role Pool

In [10]:
def retrieve_pool(queries: List[str], pool_type: str, cap_chunks: int = 6) -> List[Document]:
    """
    pool_type: 'student' or 'mentor' to enforce dataset constraints.
    """
    # Wider retrieval for mentor before filtering
    k = 10 if pool_type == "mentor" else 3
    local_retriever = vectorstore.as_retriever(search_kwargs={"k": k})

    all_docs = []
    for q in queries:
        docs = local_retriever.invoke(q)
        all_docs.extend(docs)

    all_docs = dedupe_docs(all_docs)

    # enforce pool constraint
    if pool_type == "mentor":
        all_docs = [d for d in all_docs if is_mentor_chunk(d)]
        return all_docs[:cap_chunks]  # mentor: simple cap
    else:
        all_docs = [d for d in all_docs if is_student_chunk(d)]

    # student pool: candidate-aware capping
    by_cand = defaultdict(list)
    for d in all_docs:
        sk = get_student_key(d) or "UNKNOWN"
        by_cand[sk].append(d)

    ordered_keys = [k for k in by_cand.keys() if k != "UNKNOWN"] + (["UNKNOWN"] if "UNKNOWN" in by_cand else [])
    capped = []
    i = 0
    while len(capped) < cap_chunks and any(by_cand[k] for k in ordered_keys):
        kkey = ordered_keys[i % len(ordered_keys)]
        if by_cand[kkey]:
            capped.append(by_cand[kkey].pop(0))
        i += 1

    return capped


# Run Retrieval for the 3 Pools

In [11]:
def build_role_pools(user_query: str) -> Dict[str, Any]:
    roles = infer_roles(user_query)
    subs = build_subqueries(user_query, roles)

    q1_list = rephrase_queries(subs["Q1"])
    q2_list = rephrase_queries(subs["Q2"])
    # Mentor queries must be anchored to Mentors list vocabulary
    project_context = " ".join(subs["project_keywords"][:10])
    q3_list = rephrase_mentor_queries(project_context=project_context, mentor_label=roles["mentor"])

    # Add a deterministic query that almost always hits Mentors.pdf if indexed
    q3_list = list(dict.fromkeys(q3_list + [f"Mentors list {roles['mentor']} expertise {project_context}"]))



    E1 = retrieve_pool(q1_list, pool_type="student", cap_chunks=6)
    E2 = retrieve_pool(q2_list, pool_type="student", cap_chunks=6)
    Em = retrieve_pool(q3_list, pool_type="mentor", cap_chunks=6)

    return {
        "roles": roles,
        "subqueries": subs,
        "queries": {"role1": q1_list, "role2": q2_list, "mentor": q3_list},
        "pools": {"E_student1": E1, "E_student2": E2, "E_mentor": Em},
    }

bundle = build_role_pools("Road Monitoring with Raspberry Pi. Find me 2 students and a guide.")
print(bundle["roles"])
print("E_student1:", len(bundle["pools"]["E_student1"]))
print("E_student2:", len(bundle["pools"]["E_student2"]))
print("E_mentor:", len(bundle["pools"]["E_mentor"]))


{'role1': 'Raspberry Pi, sensors, data collection', 'role2': 'image processing, computer vision, software development', 'mentor': 'IoT, embedded systems, project management'}
E_student1: 6
E_student2: 5
E_mentor: 2


# [E] Retrieval Health Check + Candidate Listing (Code)

In [12]:
def pool_candidates_student(pool: List[Document]) -> List[str]:
    keys = []
    for d in pool:
        sk = get_student_key(d)
        if sk and sk not in keys:
            keys.append(sk)
    return keys

def health_check(bundle: Dict[str, Any]) -> Dict[str, Any]:
    E1 = bundle["pools"]["E_student1"]
    E2 = bundle["pools"]["E_student2"]
    Em = bundle["pools"]["E_mentor"]

    return {
        "student1_chunks": len(E1),
        "student2_chunks": len(E2),
        "mentor_chunks": len(Em),
        "student1_candidates": pool_candidates_student(E1),
        "student2_candidates": pool_candidates_student(E2),
    }

hc = health_check(bundle)
hc


{'student1_chunks': 6,
 'student2_chunks': 5,
 'mentor_chunks': 2,
 'student1_candidates': ['David', 'Ahmed', 'Rahul'],
 'student2_candidates': ['Aisha', 'Ahmed', 'Kenji']}

# [F] Candidate Scoring & Selection (Code)

In [13]:
def tokens(s: str) -> List[str]:
    return re.findall(r"[a-zA-Z0-9]+", (s or "").lower())

def student_overlap_score(student_key: str, role_terms: List[str], project_terms: List[str]) -> int:
    meta = student_profile_map.get(student_key, {})
    kw = meta.get("keywords", []) or []
    title = meta.get("project_title", "") or ""
    dept = meta.get("department", "") or ""

    hay = set([t.lower() for t in kw] + tokens(title) + tokens(dept))
    needle = set([t.lower() for t in role_terms + project_terms])

    return len([t for t in needle if t in hay])

def score_student_candidates(pool: List[Document], role_label: str, project_terms: List[str]) -> Dict[str, float]:
    # chunk frequency per candidate
    freq = Counter()
    for d in pool:
        sk = get_student_key(d)
        if sk:
            freq[sk] += 1

    role_terms = tokens(role_label)
    scores = {}
    for sk, f in freq.items():
        overlap = student_overlap_score(sk, role_terms, project_terms)
        score = f * (1 + overlap)
        scores[sk] = float(score)
    return scores

def select_top_student(pool: List[Document], role_label: str, project_terms: List[str], banned: set[str] | None = None) -> Tuple[str | None, float]:
    banned = banned or set()
    scores = score_student_candidates(pool, role_label, project_terms)
    # remove banned
    for b in list(banned):
        scores.pop(b, None)
    if not scores:
        return None, 0.0
    sk, sc = max(scores.items(), key=lambda x: x[1])
    if sc <= 0:
        return None, 0.0
    return sk, sc


# Mentor Selection (Evidence-based only) (

In [14]:
def extract_mentor_name_from_chunk(text: str) -> str | None:
    text = clean_mentor_text(text)  # ← ADD THIS LINE, fixes the \\n \\n artifact

    m = re.search(r"\b(Dr\.|Prof\.|Professor)\s+[A-Z][a-zA-Z]+\s+[A-Z][a-zA-Z]+\b", text)
    if m:
        return m.group(0)

    m2 = re.search(r"\bName\s*:\s*([A-Z][a-zA-Z]+(?:\s+[A-Z][a-zA-Z]+){1,3})\b", text)
    if m2:
        return m2.group(1).strip()

    for ln in text.splitlines():
        ln = ln.strip()
        if 0 < len(ln) < 80:
            m3 = re.match(r"^([A-Z][a-zA-Z]+(?:\s+[A-Z][a-zA-Z]+){1,2})$", ln)
            if m3:
                return m3.group(1)
    return None

def score_mentor_candidates(pool: List[Document], mentor_label: str, project_terms: List[str]) -> Dict[str, float]:
    freq = Counter()
    overlaps = defaultdict(int)
    needle = set(tokens(mentor_label) + project_terms)

    for d in pool:
        txt = clean_mentor_text(safe_text(d, 2000))  # ← CLEAN BEFORE PROCESSING
        name = extract_mentor_name_from_chunk(txt) or "UNKNOWN_MENTOR"
        freq[name] += 1
        hay = set(tokens(txt))
        overlaps[name] += len([t for t in needle if t in hay])

    scores = {}
    for name, f in freq.items():
        score = f * (1 + overlaps[name])
        scores[name] = float(score)

    if "UNKNOWN_MENTOR" in scores and len(scores) > 1:
        scores.pop("UNKNOWN_MENTOR", None)
    return scores

def select_top_mentor(pool: List[Document], mentor_label: str, project_terms: List[str]) -> Tuple[str | None, float]:
    scores = score_mentor_candidates(pool, mentor_label, project_terms)
    if not scores:
        return None, 0.0
    name, sc = max(scores.items(), key=lambda x: x[1])
    if sc <= 0:
        return None, 0.0
    return name, sc


# [E] Cross-Pool Deduplication + Final Picks (Code)

In [15]:
def pick_team_and_mentor(bundle: Dict[str, Any]) -> Dict[str, Any]:
    roles = bundle["roles"]
    project_terms = bundle["subqueries"]["project_keywords"]

    E1 = bundle["pools"]["E_student1"]
    E2 = bundle["pools"]["E_student2"]
    Em = bundle["pools"]["E_mentor"]

    # health: if pool empty, keep None
    s1, s1_score = (None, 0.0) if len(E1) == 0 else select_top_student(E1, roles["role1"], project_terms)
    s2, s2_score = (None, 0.0) if len(E2) == 0 else select_top_student(E2, roles["role2"], project_terms)
    m, m_score   = (None, 0.0) if len(Em) == 0 else select_top_mentor(Em, roles["mentor"], project_terms)

    # cross-pool dedupe for students
    if s1 and s2 and s1 == s2:
        # keep higher score; re-pick the other excluding the kept one
        if s1_score >= s2_score:
            s2, s2_score = select_top_student(E2, roles["role2"], project_terms, banned={s1})
        else:
            s1, s1_score = select_top_student(E1, roles["role1"], project_terms, banned={s2})

    # if still same or missing, allow None
    if s1 and s2 and s1 == s2:
        s2, s2_score = None, 0.0

    return {
        "student1_key": s1, "student1_score": s1_score,
        "student2_key": s2, "student2_score": s2_score,
        "mentor_name": m, "mentor_score": m_score,
    }

picks = pick_team_and_mentor(bundle)
picks


{'student1_key': 'David',
 'student1_score': 4.0,
 'student2_key': 'Ahmed',
 'student2_score': 6.0,
 'mentor_name': 'Dr. Sarah Chen',
 'mentor_score': 3.0}

# [G] Evidence Extraction (2–4 bullets) (Code)

In [16]:
def student_display_name(student_key: str) -> str:
    meta = student_profile_map.get(student_key, {})
    return meta.get("name") or student_key

def evidence_bullets_for_student(student_key: str, pool: List[Document], max_bullets: int = 4) -> List[str]:
    # prioritize summary, then metadata, then report
    buckets = {"student_summary": [], "student_metadata": [], "student_report": [], "other": []}
    for d in pool:
        if get_student_key(d) != student_key:
            continue
        cat = (d.metadata.get("category") or "").lower()
        if "summary" in cat:
            buckets["student_summary"].append(d)
        elif "metadata" in cat:
            buckets["student_metadata"].append(d)
        elif "report" in cat:
            buckets["student_report"].append(d)
        else:
            buckets["other"].append(d)

    ordered = buckets["student_summary"] + buckets["student_metadata"] + buckets["student_report"] + buckets["other"]

    bullets = []
    for d in ordered:
        txt = safe_text(d, 500)
        # simple bullet extraction: take 1–2 sentences
        sent = re.split(r"(?<=[.!?])\s+", txt)
        for s in sent:
            s = s.strip()
            if len(s) < 30:
                continue
            bullets.append(s)
            if len(bullets) >= max_bullets:
                return bullets
    return bullets[:max_bullets]

def evidence_bullets_for_mentor(mentor_name: str, pool: List[Document], max_bullets: int = 4) -> List[str]:
    # Clean the name itself — it may have been extracted from dirty text
    mentor_name_clean = " ".join(mentor_name.split()) if mentor_name else ""

    bullets = []
    for d in pool:
        txt = clean_mentor_text(safe_text(d, 2000))  # ← CLEAN HERE
        if mentor_name_clean and mentor_name_clean in txt:
            lines = [ln.strip() for ln in txt.splitlines() if ln.strip()]
            for ln in lines:
                if mentor_name_clean in ln or any(
                    k in ln.lower() for k in ["expert", "research", "computer", "vision",
                                               "data", "systems", "iot", "nlp", "embedded", "ai"]
                ):
                    bullets.append(ln)
                    if len(bullets) >= max_bullets:
                        return bullets

    # fallback: first meaningful sentences from pool
    for d in pool:
        txt = clean_mentor_text(safe_text(d, 1200))  # ← CLEAN HERE TOO
        sent = re.split(r"(?<=[.!?])\s+", txt)
        for s in sent:
            s = s.strip()
            if len(s) < 30:
                continue
            bullets.append(s)
            if len(bullets) >= max_bullets:
                return bullets
    return bullets[:max_bullets]

# [H] Final Composer (Code)

In [17]:
def compose_answer(user_query: str, bundle: Dict[str, Any], picks: Dict[str, Any]) -> str:
    roles = bundle["roles"]
    E1 = bundle["pools"]["E_student1"]
    E2 = bundle["pools"]["E_student2"]
    Em = bundle["pools"]["E_mentor"]

    s1 = picks["student1_key"]
    s2 = picks["student2_key"]
    m  = picks["mentor_name"]

    def slot_or_not_found(x):
        return x if x else "Not found in the provided data."

    # Reasons (short, grounded-ish using metadata keywords/title)
    def student_reason(sk: str, role_label: str) -> str:
        meta = student_profile_map.get(sk, {})
        title = meta.get("project_title", "")
        kws = meta.get("keywords", [])
        kws_str = ", ".join(kws[:6]) if isinstance(kws, list) else ""
        if title or kws_str:
            return f"Aligned to **{role_label}** via project focus: {title} (keywords: {kws_str}).".strip()
        return f"Aligned to **{role_label}** based on retrieved evidence."

    def mentor_reason(name: str) -> str:
        return f"Recommended based on matching expertise evidence from Mentors.pdf."

    # Evidence
    ev = {}
    if s1 and s1 != "Not found in the provided data.":
        ev["s1"] = evidence_bullets_for_student(s1, E1, max_bullets=4)
    else:
        ev["s1"] = []
    if s2 and s2 != "Not found in the provided data.":
        ev["s2"] = evidence_bullets_for_student(s2, E2, max_bullets=4)
    else:
        ev["s2"] = []
    if m and m != "Not found in the provided data.":
        ev["m"] = evidence_bullets_for_mentor(m, Em, max_bullets=4)
    else:
        ev["m"] = []

    # Validation: mentor must have mentor pool evidence
    if m and len(Em) == 0:
        m = None

    # Final formatting
    out = []
    out.append("### 1) Team Recommendation")
    if s1:
        out.append(f"- **{student_display_name(s1)}** — **{roles['role1']}** — {student_reason(s1, roles['role1'])}")
    else:
        out.append(f"- {slot_or_not_found(None)}")

    if s2:
        out.append(f"- **{student_display_name(s2)}** — **{roles['role2']}** — {student_reason(s2, roles['role2'])}")
    else:
        out.append(f"- {slot_or_not_found(None)}")

    out.append("\n### 2) Mentor Recommendation")
    if m:
        out.append(f"- **{m}** — {mentor_reason(m)}")
    else:
        out.append(f"- {slot_or_not_found(None)}")

    out.append("\n### 3) Evidence")
    out.append(f"- **{student_display_name(s1) if s1 else 'Student 1'}**")
    if ev["s1"]:
        out.extend([f"  - {b}" for b in ev["s1"][:4]])
    else:
        out.append("  - Not found in the provided data.")

    out.append(f"- **{student_display_name(s2) if s2 else 'Student 2'}**")
    if ev["s2"]:
        out.extend([f"  - {b}" for b in ev["s2"][:4]])
    else:
        out.append("  - Not found in the provided data.")

    out.append(f"- **{m if m else 'Mentor'}**")
    if ev["m"]:
        out.extend([f"  - {b}" for b in ev["m"][:4]])
    else:
        out.append("  - Not found in the provided data.")

    return "\n".join(out)

print(compose_answer(uq, bundle, picks))


### 1) Team Recommendation
- **David Miller** — **Raspberry Pi, sensors, data collection** — Aligned to **Raspberry Pi, sensors, data collection** via project focus: Real-time Air Quality Monitoring Network with Predictive Analytics (keywords: environmental monitoring, air quality, IoT, predictive analytics).
- **Ahmed Hassan** — **image processing, computer vision, software development** — Aligned to **image processing, computer vision, software development** via project focus: AI-Optimized Traffic Flow Management System (keywords: smart cities, traffic optimization, urban planning, AI).

### 2) Mentor Recommendation
- **Dr. Sarah Chen** — Recommended based on matching expertise evidence from Mentors.pdf.

### 3) Evidence
- **David Miller**
  - combining IoT sensor technology with deep learning for critical environmental applications.
  - Future work will focus on two major areas:
1.Edge Computing Integration: Migrating the initial data cleaning and basic calibration
tasks directly on

#One “End-to-End” Run Cell (Code)

This is the only cell professors need to run after setup.

In [18]:
USER_QUERY = "I have a project Road Monitoring with Raspberry Pi. Find me 2 students and a guide."

bundle = build_role_pools(USER_QUERY)
picks = pick_team_and_mentor(bundle)
answer = compose_answer(USER_QUERY, bundle, picks)

print("ROLES:", bundle["roles"])
print("\n---\n")
print(answer)


ROLES: {'role1': 'Raspberry Pi, sensor integration, data collection', 'role2': 'data analysis, machine learning, visualization', 'mentor': 'IoT, embedded systems, environmental monitoring'}

---

### 1) Team Recommendation
- **Rahul Sharma** — **Raspberry Pi, sensor integration, data collection** — Aligned to **Raspberry Pi, sensor integration, data collection** via project focus: IoT-Based Smart Irrigation System with Crop Health Monitoring (keywords: IoT, precision agriculture, sensors, water conservation).
- **David Miller** — **data analysis, machine learning, visualization** — Aligned to **data analysis, machine learning, visualization** via project focus: Real-time Air Quality Monitoring Network with Predictive Analytics (keywords: environmental monitoring, air quality, IoT, predictive analytics).

### 2) Mentor Recommendation
- **Dr. Sarah Chen** — Recommended based on matching expertise evidence from Mentors.pdf.

### 3) Evidence
- **Rahul Sharma**
  - 4 Implementation and Depl

# Debug Cell (optional but useful during training)

Shows generated queries + top metadata so professors can see retrieval quality.

In [19]:
print("Generated Queries:")
for k,v in bundle["queries"].items():
    print(f"\n{k.upper()}:")
    for q in v:
        print(" -", q)

def show_pool(name, pool):
    print(f"\n{name} (n={len(pool)}):")
    for d in pool[:5]:
        md = d.metadata
        print(" -", md.get("source"), "|", md.get("category"), "|", md.get("student_key") or md.get("student"))

show_pool("E_student1", bundle["pools"]["E_student1"])
show_pool("E_student2", bundle["pools"]["E_student2"])
show_pool("E_mentor", bundle["pools"]["E_mentor"])


Generated Queries:

ROLE1:
 - Raspberry Pi, sensor integration, data collection road monitoring raspberry
 - Raspberry Pi sensor integration for road monitoring
 - Data collection techniques using Raspberry Pi for traffic sensors
 - Road monitoring solutions with Raspberry Pi and sensor data collection

ROLE2:
 - data analysis, machine learning, visualization road monitoring raspberry
 - road monitoring data analysis using machine learning and visualization
 - visualization techniques for raspberry pi in road monitoring data analysis
 - machine learning applications for road monitoring and data visualization with Raspberry Pi

MENTOR:
 - mentor expertise in IoT and embedded systems
 - guide for environmental monitoring projects
 - supervision in road monitoring technologies
 - mentor support for Raspberry Pi applications
 - mentor expertise IoT, embedded systems, environmental monitoring road monitoring raspberry
 - Mentors list IoT, embedded systems, environmental monitoring expertise